# Business Problem
An online retail store has a large volume of transactional data covering more than one year of sales. However, raw transaction data alone does not provide a clear overview of the store's sales performance.

A systematic analysis is needed to identify sales trends over time, determine which products and countries contribute most to revenue, and assess the overall performance of the store during the available period. Understanding these patterns can help identify the strongest and weakest areas of the business and provide useful insights for decision-making.

## Objective 
Analyze the behavior of an online store's sales during the available period, identifying the products and periods with greatest and lowest performance, and quantifying the generated revenue.

## Business Questions
1. How did sales evolve over time?
1. Which products had the best and worst performance?
1. What was the total revenue generated during the period?
1. Which countries accounted for the majority of sales?
1. What was the impact of canceled sales?


### Metrics

| Business Question | Metric | Definition |
|---|---|---|
| How did sales evolve over time? | Monthly revenue | Sum of revenue from effective sales for each month |
| How did sales evolve over time? | Monthly units sold | Sum of positive quantities from effective sales for each month |
| How did sales evolve over time? | Effective invoices | Number of unique invoices containing at least one effective sale for each month |
| Which products had the best and worst performance? | Revenue per product | Total revenue generated by each product during the available period |
| Which products had the best and worst performance? | Units sold per product | Total units sold for each product during the available period |
| What was the total revenue generated during the period? | Total revenue | Sum of revenue from all effective sales during the available period |
| Which countries accounted for the majority of sales? | Revenue by country and period | Total revenue generated by each country over time |
| Which countries accounted for the majority of sales? | Units by country and period | Total units sold by each country over time |
| What was the impact of canceled sales? | Number of cancellations | Number of invoices identified as cancellations |
| What was the impact of canceled sales? | Canceled units | Total number of units involved in canceled sales |
| What was the impact of canceled sales? | Canceled value | Total monetary value associated with canceled sales |

### Population and Filtering Criteria

- **Effective sales:** transactions with `Quantity > 0` and `Price > 0`.
- **Cancellations:** transactions whose `Invoice` starts with `C`.
- **Inventory adjustments:** excluded during the data cleaning stage.
- **Negative-price records:** excluded as identified bad-debt adjustments.
- **Zero-price transactions:** retained in the dataset but excluded from effective sales and revenue calculations.

In [1]:
import pandas as pd
import plotly.express as px

In [2]:
f = '../data/online_retail_clean.csv'
df = pd.read_csv(f)
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 516472 entries, 0 to 516471
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      516472 non-null  str    
 1   StockCode    516472 non-null  str    
 2   Description  515371 non-null  str    
 3   Quantity     516472 non-null  int64  
 4   InvoiceDate  516472 non-null  str    
 5   Price        516472 non-null  float64
 6   Customer ID  410763 non-null  float64
 7   Country      516472 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 66.2 MB


# Monthly Analysis

In [4]:
#Take only rows with Quantity and Price greater than 0
sales = df[(df["Quantity"] > 0) & (df["Price"] > 0)].copy()
sales.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [5]:
sales.shape

(504731, 8)

In [6]:
#Create the Revenue column
sales["Revenue"] = sales["Quantity"] * sales["Price"]
sales.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [7]:
sales.info()

<class 'pandas.DataFrame'>
Index: 504731 entries, 0 to 516471
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      504731 non-null  str    
 1   StockCode    504731 non-null  str    
 2   Description  504731 non-null  str    
 3   Quantity     504731 non-null  int64  
 4   InvoiceDate  504731 non-null  str    
 5   Price        504731 non-null  float64
 6   Customer ID  400916 non-null  float64
 7   Country      504731 non-null  str    
 8   Revenue      504731 non-null  float64
dtypes: float64(3), int64(1), str(5)
memory usage: 72.7 MB


In [8]:
sales["Revenue"].describe()

count    504731.000000
mean         20.351705
std          91.510976
min           0.001000
25%           4.200000
50%          10.200000
75%          17.700000
max       25111.090000
Name: Revenue, dtype: float64

In [9]:
sales[sales["Revenue"] <= 0].count()

Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
Revenue        0
dtype: int64

In [10]:
sales["InvoiceDate"] = pd.to_datetime(sales["InvoiceDate"])
sales["InvoiceDate"].dtype

dtype('<M8[us]')

In [11]:
monthly_revenue = sales.set_index("InvoiceDate").resample("ME")[["Revenue"]].sum()
monthly_revenue


,Revenue
InvoiceDate,
2009-12-31,822483.950
2010-01-31,651155.112
2010-02-28,551878.296
2010-03-31,830915.261
2010-04-30,678875.252
2010-05-31,657705.500
2010-06-30,749537.310
2010-07-31,648810.270
2010-08-31,695251.910


The maximum was in November 2010 with 1464293.142. The minimum was in February 2010 with 551878.296. Although December 2010 was less than the minimum it doesn't count because that Revenue doesn't represent the complete month.

In [12]:
monthly_units = sales.set_index("InvoiceDate").resample("ME")[["Quantity"]].sum()
monthly_units

,Quantity
InvoiceDate,
2009-12-31,425461
2010-01-31,390672
2010-02-28,381880
2010-03-31,526026
2010-04-30,366705
2010-05-31,395868
2010-06-30,406820
2010-07-31,337895
2010-08-31,472380


The maximum was in November 2010 with 725161. The minimum was in July 2010 with 337895. Although December 2010 was less than the minimum it doesn't count because that Units don't represent the complete month.

In [13]:
monthly_invoices = sales.set_index("InvoiceDate").resample("ME")[["Invoice"]].nunique()
monthly_invoices.columns = ["UniqueInvoices"]
monthly_invoices

,UniqueInvoices
InvoiceDate,
2009-12-31,1682
2010-01-31,1105
2010-02-28,1202
2010-03-31,1681
2010-04-30,1462
2010-05-31,1500
2010-06-30,1645
2010-07-31,1529
2010-08-31,1425


The maximum was in November 2010 with 72747. The minimum was in January 2010 with 1105. Although December 2010 was less than the minimum it doesn't count because that Invoices don't represent the complete month.

In [14]:
monthly_performance = pd.concat([monthly_revenue,monthly_units,monthly_invoices],axis = 1)
monthly_performance

,Revenue,Quantity,UniqueInvoices
InvoiceDate,,,
2009-12-31,822483.950,425461,1682
2010-01-31,651155.112,390672,1105
2010-02-28,551878.296,381880,1202
2010-03-31,830915.261,526026,1681
2010-04-30,678875.252,366705,1462
2010-05-31,657705.500,395868,1500
2010-06-30,749537.310,406820,1645
2010-07-31,648810.270,337895,1529
2010-08-31,695251.910,472380,1425


In [15]:
complete_months = monthly_performance[
    monthly_performance.index < "2010-12-01"
]

In [16]:
print(f"The month with the lowest revenue was: {complete_months['Revenue'].idxmin()}, "
      f"and the revenue was: {complete_months['Revenue'].min():,.2f}")

print(f"The month with the highest revenue was: {complete_months['Revenue'].idxmax()}, "
      f"and the revenue was: {complete_months['Revenue'].max():,.2f}")



The month with the lowest revenue was: 2010-02-28 00:00:00, and the revenue was: 551,878.30
The month with the highest revenue was: 2010-11-30 00:00:00, and the revenue was: 1,464,293.14


In [17]:
print(f"The month with the lowest number of units sold was: {complete_months['Quantity'].idxmin()}, "
      f"and the number of units was: {complete_months['Quantity'].min():,}")

print(f"The month with the highest number of units sold was: {complete_months['Quantity'].idxmax()}, "
      f"and the number of units was: {complete_months['Quantity'].max():,}")

The month with the lowest number of units sold was: 2010-07-31 00:00:00, and the number of units was: 337,895
The month with the highest number of units sold was: 2010-11-30 00:00:00, and the number of units was: 725,161


In [18]:
print(f"The month with the lowest number of invoices was: {complete_months['UniqueInvoices'].idxmin()}, "
      f"and the number of invoices was: {complete_months['UniqueInvoices'].min():,}")

print(f"The month with the highest number of invoices was: {complete_months['UniqueInvoices'].idxmax()}, "
      f"and the number of invoices was: {complete_months['UniqueInvoices'].max():,}")

The month with the lowest number of invoices was: 2010-01-31 00:00:00, and the number of invoices was: 1,105
The month with the highest number of invoices was: 2010-11-30 00:00:00, and the number of invoices was: 2,747


In [19]:
fig = px.line(monthly_performance, x = monthly_performance.index ,y="Revenue", 
              title = 'Monthly Revenue', markers=True)
fig.update_xaxes(title="Month")
fig.update_yaxes(title="Revenue (£)")
fig.show()
fig.write_image("../images/monthly_revenue.png")

In [20]:
fig = px.line(monthly_performance, x = monthly_performance.index ,y="Quantity", 
              title = 'Monthly Units Sold', markers=True)
fig.update_xaxes(title="Month")
fig.update_yaxes(title="Units Sold")
fig.show()

In [21]:
fig = px.line(monthly_performance, x = monthly_performance.index ,y="UniqueInvoices",
              title = 'Monthly Unique Invoices', markers=True)
fig.update_xaxes(title="Month")
fig.update_yaxes(title="Unique Invoices")
fig.show()

In [22]:
avg_units_per_invoice = monthly_performance["Quantity"] / monthly_performance["UniqueInvoices"]
print(f"The average of units sold per invoice is: {avg_units_per_invoice}")

The average of units sold per invoice is: InvoiceDate
2009-12-31    252.949465
2010-01-31    353.549321
2010-02-28    317.703827
2010-03-31    312.924450
2010-04-30    250.824213
2010-05-31    263.912000
2010-06-30    247.306991
2010-07-31    220.990844
2010-08-31    331.494737
2010-09-30    317.489940
2010-10-31    269.446328
2010-11-30    263.982890
2010-12-31    216.091127
Freq: ME, dtype: float64


In [23]:
revenue_per_unit = monthly_performance["Revenue"] / monthly_performance["Quantity"]
print(f"The revenue per units sold is: {revenue_per_unit}")

The revenue per units sold is: InvoiceDate
2009-12-31    1.933159
2010-01-31    1.666757
2010-02-28    1.445162
2010-03-31    1.579609
2010-04-30    1.851284
2010-05-31    1.661426
2010-06-30    1.842430
2010-07-31    1.920154
2010-08-31    1.471806
2010-09-30    1.578616
2010-10-31    1.874048
2010-11-30    2.019266
2010-12-31    2.428316
Freq: ME, dtype: float64


### Conclusions

- Monthly revenue decreased from December 2009 to February 2010, reaching its lowest value among complete months in February (£551,878). From March to August, revenue fluctuated without a clear upward or downward trend. It then increased significantly from September to November, reaching its highest value in November (£1,464,293).

- The number of units sold followed a similar pattern. After fluctuating between approximately 337,000 and 526,000 units during the first months of 2010, sales increased substantially from August to November, reaching a maximum of 725,161 units in November.

- The number of effective invoices was relatively stable between December 2009 and August 2010, apart from a decrease in January and February. It then increased considerably from September to November, reaching a maximum of 2,747 invoices in November.

- January and February had relatively high numbers of units sold per invoice, but their revenue per unit was comparatively low. This indicates that selling a larger number of units did not necessarily result in higher revenue during these months.

- November showed the strongest overall performance among complete months, combining the highest revenue, number of units sold, and number of effective invoices. The increase in revenue during this period was therefore associated with both a higher number of transactions and a higher revenue per unit.

- December 2010 shows lower total revenue, units sold, and effective invoices than November; however, this month is incomplete because the dataset only contains transactions up to December 9. Therefore, December should not be directly compared with the complete months.

In [24]:
sales.info()

<class 'pandas.DataFrame'>
Index: 504731 entries, 0 to 516471
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      504731 non-null  str           
 1   StockCode    504731 non-null  str           
 2   Description  504731 non-null  str           
 3   Quantity     504731 non-null  int64         
 4   InvoiceDate  504731 non-null  datetime64[us]
 5   Price        504731 non-null  float64       
 6   Customer ID  400916 non-null  float64       
 7   Country      504731 non-null  str           
 8   Revenue      504731 non-null  float64       
dtypes: datetime64[us](1), float64(3), int64(1), str(4)
memory usage: 63.5 MB


# Product Analysis

In [25]:
sales.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [26]:
product_revenue = sales.groupby(["StockCode","Description"]).agg(
    Total_Revenue = ('Revenue','sum')
)
product_revenue = product_revenue.sort_values(by ='Total_Revenue', ascending = False)
product_revenue.head(10)

,,Total_Revenue
StockCode,Description,
M,Manual,262950.93
22423,REGENCY CAKESTAND 3 TIER,169912.76
85123A,WHITE HANGING HEART T-LIGHT HOLDER,158305.72
DOT,DOTCOM POSTAGE,116408.71
84879,ASSORTED COLOUR BIRD ORNAMENT,72890.19
22086,PAPER CHAIN KIT 50'S CHRISTMAS,58127.30
85099B,JUMBO BAG RED RETROSPOT,54368.82
47566,PARTY BUNTING,49664.12
POST,POSTAGE,49477.54


In [27]:
sales[sales["StockCode"].isin(["M", "DOT", "POST"])]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
89,489439,POST,POSTAGE,3,2009-12-01 09:28:00,18.00,12682.0,France,54.00
126,489444,POST,POSTAGE,1,2009-12-01 09:55:00,141.00,12636.0,USA,141.00
173,489447,POST,POSTAGE,1,2009-12-01 10:10:00,130.00,12362.0,Belgium,130.00
613,489526,POST,POSTAGE,6,2009-12-01 11:50:00,18.00,12533.0,Germany,108.00
1207,489557,POST,POSTAGE,4,2009-12-01 12:52:00,18.00,12490.0,France,72.00
...,...,...,...,...,...,...,...,...,...
515794,538147,M,Manual,1,2010-12-09 16:11:00,15.00,13090.0,United Kingdom,15.00
515905,538148,DOT,DOTCOM POSTAGE,1,2010-12-09 16:26:00,547.32,NaN,United Kingdom,547.32
516018,538149,DOT,DOTCOM POSTAGE,1,2010-12-09 16:27:00,620.68,NaN,United Kingdom,620.68
516144,538153,DOT,DOTCOM POSTAGE,1,2010-12-09 16:31:00,822.94,NaN,United Kingdom,822.94


In [28]:
sales[sales["StockCode"].astype(str).str.contains(r"[A-Za-z]", regex=True)]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.00
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.00
12,489436,48173C,DOOR MAT BLACK FLOCK,10,2009-12-01 09:06:00,5.95,13078.0,United Kingdom,59.50
23,489436,35004B,SET OF 3 BLACK FLYING DUCKS,12,2009-12-01 09:06:00,4.65,13078.0,United Kingdom,55.80
28,489436,84596F,SMALL MARSHMALLOWS PINK BOWL,8,2009-12-01 09:06:00,1.25,13078.0,United Kingdom,10.00
...,...,...,...,...,...,...,...,...,...
516398,538170,84029E,RED WOOLLY HOTTIE WHITE HEART.,2,2010-12-09 19:32:00,3.75,13969.0,United Kingdom,7.50
516399,538170,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,2,2010-12-09 19:32:00,3.75,13969.0,United Kingdom,7.50
516400,538170,85232B,SET OF 3 BABUSHKA STACKING TINS,2,2010-12-09 19:32:00,4.95,13969.0,United Kingdom,9.90
516446,538171,47591D,PINK FAIRY CAKE CHILDRENS APRON,1,2010-12-09 20:01:00,1.95,17530.0,United Kingdom,1.95


In [29]:
sales[sales["StockCode"].astype(str).str.contains(r"[A-Za-z]", regex=True)]["StockCode"].value_counts().head(30)

StockCode
85123A     3375
85099B     2000
82494L     1134
85099F     1056
85099C      973
84970S      917
84029E      815
POST        759
47591D      732
DOT         730
84997B      677
84520B      663
84970L      649
84997D      648
84997C      640
84029G      637
84406B      600
15056N      596
85014B      567
15056BL     565
M           551
85232B      539
47590B      491
85049E      467
47556B      450
47559B      440
85014A      425
47566B      411
85049A      387
84032A      376
Name: count, dtype: int64

In [30]:
special_codes = sales[sales["StockCode"].isin(["M", "DOT", "POST"])]["StockCode"].unique()
special_codes

<ArrowStringArray>
['POST', 'DOT', 'M']
Length: 3, dtype: str

In [31]:
product_sales = sales[~sales["StockCode"].isin(["POST", "DOT", "M"])]
product_sales.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


### Product Filtering

Transaction codes `POST`, `DOT`, and `M` were excluded from the product performance analysis because they represent postage or manual charges rather than physical products. These transactions were retained in the `sales` dataset because they are valid sales-related transactions.

In [32]:
product_revenue = product_sales.groupby(["StockCode","Description"]).agg(
    Total_Revenue = ('Revenue','sum')
)
product_revenue = product_revenue.sort_values(by ='Total_Revenue', ascending = False)
product_revenue.head(10)

,,Total_Revenue
StockCode,Description,
22423,REGENCY CAKESTAND 3 TIER,169912.76
85123A,WHITE HANGING HEART T-LIGHT HOLDER,158305.72
84879,ASSORTED COLOUR BIRD ORNAMENT,72890.19
22086,PAPER CHAIN KIT 50'S CHRISTMAS,58127.30
85099B,JUMBO BAG RED RETROSPOT,54368.82
47566,PARTY BUNTING,49664.12
84347,ROTATING SILVER ANGELS T-LIGHT HLDR,47954.49
15056N,EDWARDIAN PARASOL NATURAL,36251.10
21621,VINTAGE UNION JACK BUNTING,36023.71


In [33]:
product_units = product_sales.groupby(["StockCode","Description"]).agg(
    Units_Sold = ('Quantity','sum')
)
product_units = product_units.sort_values(by ='Units_Sold', ascending = False)
product_units.head(10)

,,Units_Sold
StockCode,Description,
85123A,WHITE HANGING HEART T-LIGHT HOLDER,58386
84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,54947
17003,BROCADE RING PURSE,48374
21212,PACK OF 72 RETRO SPOT CAKE CASES,46728
84879,ASSORTED COLOUR BIRD ORNAMENT,45228
84991,60 TEATIME FAIRY CAKE CASES,36348
21977,PACK OF 60 PINK PAISLEY CAKE CASES,31805
85099B,JUMBO BAG RED RETROSPOT,30327
22197,SMALL POPCORN HOLDER,29773


In [34]:
product_performance = pd.concat([product_revenue,product_units],axis = 1)
product_performance

,,Total_Revenue,Units_Sold
StockCode,Description,,
22423,REGENCY CAKESTAND 3 TIER,169912.760,13685
85123A,WHITE HANGING HEART T-LIGHT HOLDER,158305.720,58386
84879,ASSORTED COLOUR BIRD ORNAMENT,72890.190,45228
22086,PAPER CHAIN KIT 50'S CHRISTMAS,58127.300,17205
85099B,JUMBO BAG RED RETROSPOT,54368.820,30327
...,...,...,...
79151B,"SILICON CUBE 25W, BLUE",0.420,1
35930,PINK HEART CHRISTMAS DECORATION,0.380,1
84206C,CHAMPAGNE TRAY BLANK CARD,0.380,2


In [35]:
product_performance["Revenue_Per_Unit"] = product_performance["Total_Revenue"] / product_performance["Units_Sold"]
product_performance.head(10)

,,Total_Revenue,Units_Sold,Revenue_Per_Unit
StockCode,Description,,,
22423,REGENCY CAKESTAND 3 TIER,169912.76,13685,12.415985
85123A,WHITE HANGING HEART T-LIGHT HOLDER,158305.72,58386,2.711364
84879,ASSORTED COLOUR BIRD ORNAMENT,72890.19,45228,1.611616
22086,PAPER CHAIN KIT 50'S CHRISTMAS,58127.30,17205,3.378512
85099B,JUMBO BAG RED RETROSPOT,54368.82,30327,1.792753
47566,PARTY BUNTING,49664.12,10079,4.927485
84347,ROTATING SILVER ANGELS T-LIGHT HLDR,47954.49,23037,2.081629
15056N,EDWARDIAN PARASOL NATURAL,36251.10,7416,4.888228
21621,VINTAGE UNION JACK BUNTING,36023.71,4144,8.692980


In [36]:
revenue_top_10 = product_performance.sort_values(
    by="Total_Revenue",
    ascending=False
).reset_index().head(10)
fig = px.bar(
    revenue_top_10,
    x = 'Total_Revenue',
    y = 'Description',
    orientation = 'h',
    title = 'Top 10 products by Revenue',
    text_auto = '.2f'

)
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'}, 
    xaxis_title="Total Revenue",
    yaxis_title="Product",
    coloraxis_showscale=False
)

fig.show()
fig.write_image("../images/product_revenue.png")

In [37]:
units_top_10 = product_performance.sort_values(
    by="Units_Sold",
    ascending=False
).reset_index().head(10)
units_top_10

,StockCode,Description,Total_Revenue,Units_Sold,Revenue_Per_Unit
0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,158305.72,58386,2.711364
1,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,11383.97,54947,0.207181
2,17003,BROCADE RING PURSE,8922.80,48374,0.184454
3,21212,PACK OF 72 RETRO SPOT CAKE CASES,24054.43,46728,0.514776
4,84879,ASSORTED COLOUR BIRD ORNAMENT,72890.19,45228,1.611616
5,84991,60 TEATIME FAIRY CAKE CASES,18151.40,36348,0.499378
6,21977,PACK OF 60 PINK PAISLEY CAKE CASES,16187.31,31805,0.508955
7,85099B,JUMBO BAG RED RETROSPOT,54368.82,30327,1.792753
8,22197,SMALL POPCORN HOLDER,27022.23,29773,0.907609
9,21232,STRAWBERRY CERAMIC TRINKET BOX,34411.68,27059,1.271728


In [38]:
fig = px.bar(
    units_top_10,
    x = 'Units_Sold',
    y = 'Description',
    orientation = 'h',
    title = 'Top 10 by Units Sold',
    text = 'Units_Sold',
    
)
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'}, 
    xaxis_title="Units sold",
    yaxis_title="Product",
    coloraxis_showscale=False
)

fig.show()

In [39]:
product_performance.shape

(4729, 3)

In [40]:
product_performance["Units_Sold"].describe()

count     4729.000000
mean      1227.985409
std       3102.524865
min          1.000000
25%         47.000000
50%        298.000000
75%       1094.000000
max      58386.000000
Name: Units_Sold, dtype: float64

In [41]:
product_performance["Total_Revenue"].describe()

count      4729.000000
mean       2081.475799
std        5430.505127
min           0.014000
25%         122.360000
50%         566.220000
75%        1867.110000
max      169912.760000
Name: Total_Revenue, dtype: float64

In [42]:
product_performance.sort_values(
    by="Total_Revenue",
    ascending=True
).head(10)

,,Total_Revenue,Units_Sold,Revenue_Per_Unit
StockCode,Description,,,
PADS,PADS TO MATCH ALL CUSHIONS,0.014,14,0.001
84205C,HAPPY BIRTHDAY GINGER CAT CARD,0.380,2,0.190
84206C,CHAMPAGNE TRAY BLANK CARD,0.380,2,0.190
35930,PINK HEART CHRISTMAS DECORATION,0.380,1,0.380
79151B,"SILICON CUBE 25W, BLUE",0.420,1,0.420
84206B,CAT W SUNGLASSES BLANK CARD,0.760,4,0.190
79157B,UBO-LIGHT TRIOBASE BLUE,0.850,1,0.850
20721,OWLS CHARLOTTE BAG,0.850,1,0.850
44092B,BLUE WHITE PLASTIC RINGS LAMPSHADE,0.850,1,0.850


In [43]:
product_performance.sort_values(
    by="Units_Sold",
    ascending=True
).head(10)

,,Total_Revenue,Units_Sold,Revenue_Per_Unit
StockCode,Description,,,
79157B,UBO-LIGHT TRIOBASE BLUE,0.85,1,0.85
44092B,BLUE WHITE PLASTIC RINGS LAMPSHADE,0.85,1,0.85
20721,OWLS CHARLOTTE BAG,0.85,1,0.85
TEST002,This is a test product.,1.00,1,1.00
85171,S/2 HYACINTH BULB CANDLES,2.55,1,2.55
47554,S/3 COLOURS FLOCK TEA TOWELS,2.55,1,2.55
47569,ENGLISH ROSE DESIGN SHOPPING BAG,2.55,1,2.55
21008,ETCHED GLASS HEART TREE DECORATION,2.51,1,2.51
DCGS0062,ROAD-RAGE CAR FRESHENER,2.51,1,2.51


In [44]:
product_invoices = product_sales.groupby(["StockCode", "Description"]).agg(
    Unique_Invoices=("Invoice", "nunique")
)

In [45]:
product_performance = pd.concat(
    [product_performance, product_invoices],
    axis=1
)
product_performance.head()

,,Total_Revenue,Units_Sold,Revenue_Per_Unit,Unique_Invoices
StockCode,Description,,,,
22423,REGENCY CAKESTAND 3 TIER,169912.76,13685,12.415985,2019
85123A,WHITE HANGING HEART T-LIGHT HOLDER,158305.72,58386,2.711364,3281
84879,ASSORTED COLOUR BIRD ORNAMENT,72890.19,45228,1.611616,1412
22086,PAPER CHAIN KIT 50'S CHRISTMAS,58127.30,17205,3.378512,957
85099B,JUMBO BAG RED RETROSPOT,54368.82,30327,1.792753,1222


In [46]:
product_performance["Unique_Invoices"].describe()

count    4729.000000
mean      104.963206
std       169.271852
min         1.000000
25%        12.000000
50%        42.000000
75%       126.000000
max      3281.000000
Name: Unique_Invoices, dtype: float64

In [47]:
low_performance = product_performance[product_performance["Unique_Invoices"] >= 10]

In [48]:
low_performance.sort_values(
    by="Units_Sold",
    ascending=True
).head(10)

,,Total_Revenue,Units_Sold,Revenue_Per_Unit,Unique_Invoices
StockCode,Description,,,,
90054,CRACKED GLAZE EARRINGS RED,25.53,10,2.553000,10
90181B,AMETHYST GLASS/SHELL/PEARL NECKLACE,112.32,12,9.360000,12
90202C,GREEN ENAMEL FLOWER HAIR TIE,35.71,12,2.975833,12
90130C,GREEN STONE/CRYSTAL EARRINGS,35.63,12,2.969167,12
47552A,DOTS IRONING BOARD COVER,23.40,12,1.950000,11
90118,PINK DAISY BAG CHARM,32.75,12,2.729167,10
90033,IVORY SHELL HEART EARRINGS,33.36,13,2.566154,10
90174,BUTTERFLY HAIR BAND,33.24,13,2.556923,12
90164A,PINK ROSEBUD PEARL BRACELET,99.72,13,7.670769,11


In [49]:
low_performance.sort_values(
    by="Total_Revenue",
    ascending=True
).head(10)

,,Total_Revenue,Units_Sold,Revenue_Per_Unit,Unique_Invoices
StockCode,Description,,,,
PADS,PADS TO MATCH ALL CUSHIONS,0.014,14,0.001000,14
90214V,"LETTER ""V"" BLING KEY RING",15.100,14,1.078571,12
35932,PINK CHRISTMAS TREE DECORATION,15.630,37,0.422432,10
84251B,"GREETING CARD, STICKY GORDON",15.760,76,0.207368,10
84250M,"GREETING CARD, BLACKPOOL ROCK",15.900,28,0.567857,13
84993a,75 GREEN PETIT FOUR CASES,16.150,19,0.850000,11
84745A,PINK HANGING GINGHAM EASTER HEN,16.800,40,0.420000,10
84993b,75 BLACK PETIT FOUR CASES,16.990,20,0.849500,13
44089C,LILAC GAUZE BUTTERFLY LAMPSHADE,18.480,44,0.420000,12


In [50]:
pads = product_sales[product_sales["StockCode"] == "PADS"]
pads

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
61490,494914,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-01-19 17:04:00,0.001,16705.0,United Kingdom,0.001
73716,496222,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-01-29 13:53:00,0.001,13583.0,United Kingdom,0.001
76642,496473,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-02-01 15:38:00,0.001,17350.0,United Kingdom,0.001
78701,496643,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-02-03 11:58:00,0.001,13408.0,United Kingdom,0.001
89274,497935,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-02-15 10:47:00,0.001,13408.0,United Kingdom,0.001
96079,498562,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-02-21 12:03:00,0.001,15182.0,United Kingdom,0.001
99988,499056,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-02-24 13:46:00,0.001,13765.0,United Kingdom,0.001
102701,499399,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-02-26 13:26:00,0.001,14459.0,United Kingdom,0.001
121819,501176,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-03-15 11:00:00,0.001,14857.0,United Kingdom,0.001
154116,504332,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-04-12 16:30:00,0.001,12671.0,Germany,0.001


In [51]:
df[df["StockCode"] == "PADS"]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
61490,494914,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-01-19 17:04:00,0.001,16705.0,United Kingdom
73716,496222,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-01-29 13:53:00,0.001,13583.0,United Kingdom
76642,496473,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-02-01 15:38:00,0.001,17350.0,United Kingdom
78701,496643,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-02-03 11:58:00,0.001,13408.0,United Kingdom
89274,497935,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-02-15 10:47:00,0.001,13408.0,United Kingdom
96079,498562,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-02-21 12:03:00,0.001,15182.0,United Kingdom
99988,499056,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-02-24 13:46:00,0.001,13765.0,United Kingdom
102701,499399,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-02-26 13:26:00,0.001,14459.0,United Kingdom
121819,501176,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-03-15 11:00:00,0.001,14857.0,United Kingdom
154116,504332,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-04-12 16:30:00,0.001,12671.0,Germany


### Conclusions

- The **REGENCY CAKESTAND 3 TIER** generated the highest revenue during the available period. However, it did not rank among the top 10 products by units sold, indicating that high revenue was not necessarily associated with the highest sales volume.

- The **WHITE HANGING HEART T-LIGHT HOLDER** was the product with the highest number of units sold and ranked second in total revenue. This indicates that its high sales volume translated into a substantial contribution to total revenue.

- Several products with high sales volumes did not rank among the top products by revenue. This can be explained by their relatively low average prices, showing that products with high sales volumes do not necessarily generate the highest revenue.

- After filtering the products to include only those that appeared in at least 10 unique invoices, **PADS** was identified as an unusual case. It appeared in 14 invoices, with each positive transaction recorded at a price of £0.001. The original dataset also contains a cancellation for this product recorded at £36.60. Since its positive transactions meet the criteria established for effective sales, the product was retained in the analysis.

- The comparison between revenue and units sold shows that some products generated more revenue despite having lower sales volumes than other products. This indicates that sales volume alone does not determine a product's contribution to revenue, as the price at which products are sold also plays an important role.

# Geographical Analysis

In [52]:
countries = sales.groupby("Country").agg(
    Total_Revenue=("Revenue", "sum"),
    Units_Sold = ("Quantity","sum")
)
countries["Revenue_Per_Unit"] = countries["Total_Revenue"] / countries["Units_Sold"]
countries.head()

,Total_Revenue,Units_Sold,Revenue_Per_Unit
Country,,,
Australia,31446.80,20189,1.557620
Austria,13414.33,6697,2.003036
Bahrain,2355.65,1025,2.298195
Belgium,24537.58,12059,2.034794
Bermuda,1253.14,2798,0.447870


In [53]:
country_revenue = countries.sort_values(by='Total_Revenue',ascending=False).reset_index()
country_revenue.head(10)

,Country,Total_Revenue,Units_Sold,Revenue_Per_Unit
0,United Kingdom,8812685.403,4702281,1.874130
1,EIRE,380909.570,193047,1.973144
2,Netherlands,268784.350,183615,1.463847
3,Germany,202025.391,108615,1.860014
4,France,147103.140,163029,0.902313
5,Sweden,53501.990,52555,1.018019
6,Denmark,50906.850,229690,0.221633
7,Spain,47568.650,22841,2.082599
8,Switzerland,43921.390,22255,1.973552
9,Australia,31446.800,20189,1.557620


In [54]:
country_units = countries.sort_values(by='Units_Sold',ascending=False).reset_index()
country_units.head(10)

,Country,Total_Revenue,Units_Sold,Revenue_Per_Unit
0,United Kingdom,8812685.403,4702281,1.874130
1,Denmark,50906.850,229690,0.221633
2,EIRE,380909.570,193047,1.973144
3,Netherlands,268784.350,183615,1.463847
4,France,147103.140,163029,0.902313
5,Germany,202025.391,108615,1.860014
6,Sweden,53501.990,52555,1.018019
7,Spain,47568.650,22841,2.082599
8,Switzerland,43921.390,22255,1.973552
9,Australia,31446.800,20189,1.557620


In [55]:
country_revenue_unit = countries.sort_values(by='Revenue_Per_Unit',ascending=False).reset_index()
country_revenue_unit.head(10)

,Country,Total_Revenue,Units_Sold,Revenue_Per_Unit
0,Hong Kong,8202.52,2308,3.553951
1,Malta,5373.50,1549,3.469012
2,Norway,23944.18,7867,3.043623
3,Lebanon,211.70,72,2.940278
4,Israel,3199.40,1132,2.826325
5,Cyprus,11347.10,4610,2.461410
6,RSA,3802.01,1619,2.348369
7,Greece,14335.67,6167,2.324578
8,Singapore,4037.77,1753,2.303349
9,Bahrain,2355.65,1025,2.298195


In [56]:
fig = px.bar(
    country_revenue.head(10),
    x = 'Total_Revenue',
    y = 'Country',
    orientation = 'h',
    title = 'Top 10 Countries by Revenue',
    text_auto = '.2f'

)
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'}, 
    xaxis_title="Total Revenue",
    yaxis_title="Country",
    coloraxis_showscale=False
)

fig.show()
fig.write_image("../images/country_revenue.png")

In [57]:
fig = px.bar(
    country_units.head(10),
    x = 'Units_Sold',
    y = 'Country',
    orientation = 'h',
    title = 'Top 10 Countries by Units Sold',   
    text = 'Units_Sold'
)
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'}, 
    xaxis_title="Units Sold",
    yaxis_title="Country",
    coloraxis_showscale=False
)

fig.show()

In [58]:
fig = px.bar(
    country_revenue_unit.head(10),
    x = 'Revenue_Per_Unit',
    y = 'Country',
    orientation = 'h',
    title = 'Top 10 Countries by Revenue Per Unit',
    text_auto = '.2f'

)
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'}, 
    xaxis_title="Revenue Per Unit",
    yaxis_title="Country",
    coloraxis_showscale=False
)

fig.show()

# Conclusions
* The United Kingdom accounted for the vast majority of revenue and units sold during the available period, with approximately £8.81 million in revenue and 4.70 million units sold. This indicates that the sales activity in the dataset was strongly concentrated in the UK.
* The ranking of countries differs depending on whether performance is measured by revenue or units sold. For example, Denmark ranked second in units sold with 229,690 units, but generated only £50,906.85 in revenue. In contrast, Germany sold 108,615 units but generated £202,025.39 in revenue.
* Revenue per unit helps explain these differences in performance. Denmark generated approximately £0.22 per unit, while Germany generated £1.86 and Spain £2.08 per unit. Therefore, a higher sales volume did not necessarily translate into higher revenue.
* Spain generated £47,568.65 from only 22,841 units, resulting in the highest revenue per unit among the countries in the Top 10 revenue analysis. This contrasts with Denmark, which generated a similar amount of revenue from more than ten times as many units.
* Overall, the geographic analysis shows that country performance varied considerably depending on the metric used. Revenue, units sold, and revenue per unit provide different perspectives on the international sales distribution and should therefore be considered together rather than using a single metric.

# Cancellation Analysis

In [59]:
cancellations = df[df["Invoice"].astype(str).str.startswith("C")].copy()
cancellations.shape

(10182, 8)

In [60]:
cancellations[["Quantity","Price"]].describe()

,Quantity,Price
count,10182.000000,10182.000000
mean,-20.947555,41.754665
std,208.505474,524.896310
min,-9360.000000,0.010000
25%,-6.000000,1.650000
50%,-2.000000,2.950000
75%,-1.000000,6.750000
max,1.000000,25111.090000


In [61]:
cancellations[cancellations["Quantity"] > 0]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
75754,C496350,M,Manual,1,2010-02-01 08:24:00,373.57,NaN,United Kingdom


In [62]:
cancellations = cancellations[~((cancellations["StockCode"] == "M") & (cancellations["Quantity"] > 0))].copy()

In [63]:
canceled_units = cancellations["Quantity"].abs().sum()
cancellations["Canceled_Value"] = (cancellations["Quantity"].abs() * cancellations["Price"])
canceled_value = cancellations["Canceled_Value"].sum()
total_revenue = sales["Revenue"].sum()
cancellation_impact = (canceled_value / total_revenue) * 100 #Canceled value as a percentage of effective sales revenue

In [64]:
print(cancellations.shape)
print(cancellations["Invoice"].nunique())
print(canceled_units)
print(canceled_value)
print(cancellation_impact)

(10181, 9)
4591
213289
629808.6499999999
6.131233422658284


In [65]:
cancellations.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Canceled_Value
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,35.4
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia,9.9
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia,17.0
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia,12.6
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,35.4


In [66]:
cancellations["Canceled_Units"] = cancellations["Quantity"].abs()
cancellations["InvoiceDate"] = pd.to_datetime(cancellations["InvoiceDate"])
monthly_cancellations = (
    cancellations.set_index("InvoiceDate").resample("ME").agg(
        Units_Cancel=("Canceled_Units", "sum"),
        Total_Cancel=("Canceled_Value", "sum"),
        Cancellation_Invoices=("Invoice", "nunique")
        
    )
)

monthly_cancellations

,Units_Cancel,Total_Cancel,Cancellation_Invoices
InvoiceDate,,,
2009-12-31,10016,25835.45,401
2010-01-31,3880,28675.61,300
2010-02-28,6952,20612.93,239
2010-03-31,4477,67668.02,407
2010-04-30,6483,37354.20,304
2010-05-31,16599,44434.78,407
2010-06-30,15482,72463.44,357
2010-07-31,6185,31444.79,344
2010-08-31,6689,40477.52,273


In [67]:
cancellations[
    cancellations["InvoiceDate"].dt.month == 9
].sort_values("Quantity")[
    ["Invoice", "StockCode", "Description", "Quantity", "Price", "Canceled_Value"]
].head(20)

,Invoice,StockCode,Description,Quantity,Price,Canceled_Value
353747,C524235,21088,SET/6 FRUIT SALAD PAPER CUPS,-7128,0.08,570.24
353748,C524235,21096,SET/6 FRUIT SALAD PAPER PLATES,-7008,0.13,911.04
353708,C524235,16047,POP ART PEN CASE & PENS,-5184,0.08,414.72
353714,C524235,37340,MULTICOLOUR SPRING FLOWER MUG,-4992,0.10,499.20
353731,C524235,85110,BLACK SILVER FLOWER T-LIGHT HOLDER,-4752,0.07,332.64
353736,C524235,16046,TEATIME PEN CASE & PENS,-4608,0.08,368.64
353732,C524235,85160A,WHITE BIRD GARDEN DESIGN MUG,-4320,0.13,561.60
353752,C524235,85184D,S/4 BLUE ROUND DECOUPAGE BOXES,-3936,0.42,1653.12
353738,C524235,16162L,THE KING GIFT BAG,-3744,0.05,187.20
353712,C524235,21392,RED POLKADOT PUDDING BOWL,-3648,0.13,474.24


September shows an unusually high number of canceled units, largely driven by a single cancellation invoice (C524235) containing multiple product lines with exceptionally large quantities.

In [68]:
fig = px.line(monthly_cancellations, x = monthly_cancellations.index ,y="Total_Cancel", 
              title = 'Monthly Canceled Value', markers=True)
fig.update_xaxes(title="Month")
fig.update_yaxes(title="Canceled Value (£)")
fig.show()
fig.write_image("../images/cancellations.png")

### Conclusions

- A total of **£629,808.65** in canceled value was recorded during the available period.

- The total canceled value represents approximately **6.13% of the effective sales revenue** generated during the available period. This percentage compares the value associated with cancellations with the revenue from effective sales and should not be interpreted as a cancellation rate.

- **October 2010** recorded the highest canceled value, with approximately **£81,291**.

- Although **September 2010** recorded the highest number of canceled units, it did not have the highest canceled value. This indicates that the number of canceled units does not necessarily reflect the financial impact of cancellations, as the unit prices of the canceled products also influence the total value.

- Although the dataset only contains cancellation records through **December 9, 2010**, December had a canceled value of approximately **£61,364**, which was higher than several complete months. However, since December is incomplete, its total should not be directly compared with the complete months.